In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
df = df.drop('id', axis=1)


In [3]:
num_features = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OrdinalEncoder

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_features), 
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features)
])

In [5]:
from sklearn.model_selection import train_test_split 

X = df.drop('stroke', axis=1)
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
X_train_prepped = preprocessor.fit_transform(X_train)
X_test_prepped = preprocessor.transform(X_test)


In [20]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import f1_score
import numpy as np 

forest = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42)
gradient = GradientBoostingClassifier(n_estimators=500, max_depth=3, random_state=42, learning_rate=0.05)
voter = VotingClassifier(estimators=[('forest', forest), ('gradient', gradient)], voting='soft')


In [27]:
import time
from sklearn.utils.class_weight import compute_sample_weight

# 1. Compute weights
sw = compute_sample_weight(class_weight='balanced', y=y_train)

models = {
    'RandomForest': forest, 
    'GradientBoosting': gradient, 
    'VotingClassifer': voter
}
performance_records = []

# 2. The Cleaned Loop
for name, model in models.items():
    # Starting the stopwatch 
    start = time.time()
    
    # Give the sample weights uniformly to ALL models
    model.fit(X_train_prepped, y_train, sample_weight=sw)
    
    end = time.time()
    
    # Calculate the records 
    y_pred = model.predict(X_test_prepped)
    f1 = f1_score(y_test, y_pred)

    # Store data 
    performance_records.append({
        'Model_Architecture': name, 
        'Train_Time_Sec': round(end - start, 4), 
        'F1_Score': round(f1, 4)
    })

In [28]:
# Conver list of dictionaries into a df 
df_report = pd.DataFrame(performance_records)
print(df_report)

  Model_Architecture  Train_Time_Sec  F1_Score
0       RandomForest          1.0373    0.2200
1   GradientBoosting          1.5289    0.2234
2    VotingClassifer          2.5437    0.2571


In [ ]:
df_report.to_csv('ensemble_benchmark_report(1).csv', index=False)